# NyayaSetu — Restart-Safe Legal AI Notebook

**Architecture preserved:**
- MuRIL Legal Document Classifier (Hugging Face)
- NLLB-200 Translation (Hugging Face)
- BART Summarizer (Hugging Face)
- SentenceTransformer embeddings (Hugging Face)
- FAISS + BM25 Hybrid Retrieval
- FastAPI server with ngrok exposure

**Restart-safe behavior:**
- Maps Hugging Face cache to Google Drive so models are downloaded **once**.
- First run: trains, builds indexes, saves everything to Drive.
- Subsequent runs (`Runtime → Run all`): loads everything, skips training, launches server.

**Sections:**
1. Install  2. Mount Drive  3. Config & HF Cache  4. Imports  5. HF Login
6. Dataset  7. Classifier  8. NLLB  9. BART  10. SentenceTransformer
11. Embeddings  12. FAISS  13. BM25  14. Metadata  15. FastAPI  16. ngrok  17. Uvicorn  18. Testing

## 1. Install (conditional — only missing packages)

In [2]:
import importlib, subprocess, sys, os

REQUIRED = {
    'fitz': 'pymupdf',
    'pytesseract': 'pytesseract',
    'docx': 'python-docx',
    'indicnlp': 'indic-nlp-library',
    'tqdm': 'tqdm',
    'sentence_transformers': 'sentence-transformers',
    'faiss': 'faiss-cpu',
    'rank_bm25': 'rank-bm25',
    'fastapi': 'fastapi',
    'uvicorn': 'uvicorn',
    'pyngrok': 'pyngrok',
    'transformers': 'transformers',
    'huggingface_hub': 'huggingface_hub',
    'datasets': 'datasets',
    'scikit_learn': 'scikit-learn',
    'pandas': 'pandas',
    'PIL': 'pillow',
}

def _pkg_name(mod): return REQUIRED.get(mod, mod)

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(_pkg_name(mod))

if missing:
    print('Installing missing packages:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + missing, check=False)
else:
    print('All required Python packages already available.')

# Tesseract language packs (apt, idempotent)
import shutil
if not shutil.which('tesseract'):
    subprocess.run(['apt-get', 'install', '-y', 'tesseract-ocr'], check=False)
for lang in ['hin', 'mar']:
    r = subprocess.run(['tesseract', '--list-langs'], capture_output=True, text=True)
    if lang not in (r.stdout or ''):
        subprocess.run(['apt-get', 'install', '-y', f'tesseract-ocr-{lang}'], check=False)
    else:
        print(f'tesseract-ocr-{lang} already installed.')

Installing missing packages: ['pymupdf', 'pytesseract', 'python-docx', 'indic-nlp-library', 'faiss-cpu', 'rank-bm25', 'pyngrok', 'scikit-learn']


## 2. Mount Drive & Create Folder Structure

In [5]:
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

ROOT = '/content/drive/MyDrive/NyayaSetu'
SUBDIRS = [
    'models/muril', 'models/nllb', 'models/bart', 'models/sentence-transformer',
    'models/classifier', 'indexes', 'embeddings', 'metadata', 'processed', 'server', 'hf_cache'
]
for d in SUBDIRS:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)
print('Drive structure ready at:', ROOT)

Mounted at /content/drive
Drive structure ready at: /content/drive/MyDrive/NyayaSetu


## 3. Configuration & Hugging Face Cache Mapping

In [6]:
import os

# Map Hugging Face Cache to Google Drive to avoid re-downloading on restart
HF_CACHE_DIR = os.path.join(ROOT, 'hf_cache')
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')
print(f'Hugging Face cache mapped to: {HF_CACHE_DIR}')

RAW_PDF_PATH      = '/content/drive/MyDrive/NyayaSetu_Dataset/raw-pdfs'
DATASET_CSV       = os.path.join(ROOT, 'processed', 'master_dataset.csv')
MURIL_DIR         = os.path.join(ROOT, 'models', 'muril')
CLASSIFIER_DIR    = os.path.join(ROOT, 'models', 'classifier')
NLLB_DIR          = os.path.join(ROOT, 'models', 'nllb')
BART_DIR          = os.path.join(ROOT, 'models', 'bart')
ST_DIR            = os.path.join(ROOT, 'models', 'sentence-transformer')
FAISS_INDEX_PATH  = os.path.join(ROOT, 'indexes', 'faiss.index')
BM25_PATH         = os.path.join(ROOT, 'indexes', 'bm25.pkl')
EMB_PATH          = os.path.join(ROOT, 'embeddings', 'embeddings.npy')
META_PATH         = os.path.join(ROOT, 'metadata', 'metadata.json')
LABEL_MAP_PATH    = os.path.join(ROOT, 'metadata', 'label_map.json')

MURIL_BASE        = 'google/muril-base-cased'
NLLB_BASE         = 'facebook/nllb-200-distilled-600M'
BART_BASE         = 'facebook/bart-large-cnn'
ST_BASE           = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

MAX_LEN           = 512
EMB_DIM           = 384

NGROK_TOKEN       = '3HOr5UUZVbJN44Jn2FjOSiap2BP_3tzF8vkCJ19iZQT4jb1sG'  # <-- paste your ngrok authtoken here

def exists_all(paths):
    return all(os.path.exists(p) for p in paths)

def save_json(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        import json
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        import json
        return json.load(f)

Hugging Face cache mapped to: /content/drive/MyDrive/NyayaSetu/hf_cache


## 4. Imports

In [7]:
import os, re, io, gc, json, pickle, shutil, subprocess, sys, time, unicodedata
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import torch
import fitz
import pytesseract
from PIL import Image
from tqdm import tqdm
from docx import Document

from indicnlp.normalize.indic_normalize import IndicNormalizerFactory

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM, BartTokenizer, BartForConditionalGeneration,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from sentence_transformers import SentenceTransformer

import faiss
from rank_bm25 import BM25Okapi

from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok, conf

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


## 5. Hugging Face Login (Optional - for gated models)

In [ ]:
from huggingface_hub import login

# If you are using private or gated models, you need to login.
# You can generate an access token from: https://huggingface.co/settings/tokens

HF_TOKEN = "" # Paste your HF token here if needed. Leave empty if using public models.

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Successfully logged into Hugging Face Hub.")
else:
    print("Skipping Hugging Face login (using public models).")

Skipping Hugging Face login (using public models).


## 6. Dataset Creation (conditional — skips if CSV exists)

In [ ]:
factory = IndicNormalizerFactory()
_hi_norm = factory.get_normalizer('hi')
_mr_norm = factory.get_normalizer('mr')

def normalize_text(text, lang='hi'):
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[•■□▪]+', ' ', text)
    text = text.strip()
    if lang == 'hi':  text = _hi_norm.normalize(text)
    elif lang == 'mr': text = _mr_norm.normalize(text)
    return text

def detect_language(text):
    marathi_keywords = ['आहे', 'मालमत्ता', 'करण्यात', 'सदर', 'याबाबत', 'अर्जदार']
    for w in marathi_keywords:
        if w in text:
            return 'marathi'
    return 'hindi'

def perform_ocr(page):
    pix = page.get_pixmap(dpi=200)
    img = Image.open(io.BytesIO(pix.tobytes()))
    return pytesseract.image_to_string(img, lang='hin+mar+eng')

def split_bail_applications(text):
    patterns = [r'जमानत\s+अर्ज', r'CRIMINAL BAIL APPLICATION', r'जमानत\s+अर्जी']
    chunks = [text]
    for p in patterns:
        tmp = []
        for c in chunks:
            tmp.extend(re.split(p, c))
        chunks = tmp
    return [c for c in chunks if len(c.strip()) > 300]

def extract_pdf_chunks(pdf_path, label):
    chunks = []
    try:
        doc = fitz.open(pdf_path)
        pages = doc[:3] if label == 'Affidavit' else doc
        for i, page in enumerate(pages):
            text = page.get_text()
            if len(text.strip()) < 10:
                if label != 'Affidavit':
                    text = perform_ocr(page)
                else:
                    continue
            text = text.strip()
            if len(text) < 150:
                continue
            chunks.append({'text': text, 'page': i + 1})
    except Exception as e:
        print(f'PDF ERROR: {pdf_path}: {e}')
    return chunks

def extract_docx_chunks(docx_path):
    chunks = []
    try:
        doc = Document(docx_path)
        full = []
        for para in doc.paragraphs:
            t = para.text.strip()
            if len(t) > 20:
                full.append(t)
        full = '\n'.join(full)
        for idx, c in enumerate(re.split(r'\n+', full)):
            if len(c.strip()) > 150:
                chunks.append({'text': c, 'page': idx + 1})
    except Exception as e:
        print(f'DOCX ERROR: {docx_path}: {e}')
    return chunks

def build_dataset():
    rows = []
    print('Building dataset from raw PDFs...')
    for label in os.listdir(RAW_PDF_PATH):
        folder = os.path.join(RAW_PDF_PATH, label)
        if not os.path.isdir(folder):
            continue
        print(f'Processing class: {label}')
        for fn in tqdm(os.listdir(folder)):
            fp = os.path.join(folder, fn)
            try:
                chunks = []
                if fn.lower().endswith('.pdf'):
                    chunks = extract_pdf_chunks(fp, label)
                elif fn.lower().endswith(('.docx', '.doc')):
                    chunks = extract_docx_chunks(fp)
                for c in chunks:
                    text = c['text']
                    if label == 'Bail_Application':
                        splits = split_bail_applications(text)
                    else:
                        splits = [text]
                    for s in splits:
                        if len(s.strip()) < 150:
                            continue
                        lang = detect_language(s)
                        norm = normalize_text(s, 'mr' if lang == 'marathi' else 'hi')
                        rows.append({
                            'text': norm, 'label': label, 'language': lang,
                            'source_file': fn, 'page': c['page']
                        })
            except Exception as e:
                print(f'ERROR: {fn}: {e}')
    df = pd.DataFrame(rows).drop_duplicates(subset=['text']).reset_index(drop=True)
    df.to_csv(DATASET_CSV, index=False)
    print(f'Dataset saved: {df.shape} → {DATASET_CSV}')
    return df

if os.path.exists(DATASET_CSV):
    print(f'Loading dataset from Drive: {DATASET_CSV}')
    master_df = pd.read_csv(DATASET_CSV)
    print('Dataset shape:', master_df.shape)
else:
    if os.path.exists(RAW_PDF_PATH):
        master_df = build_dataset()
    else:
        print(f'WARNING: Raw PDF path not found ({RAW_PDF_PATH}).')
        print('Creating minimal placeholder dataset so downstream cells can still load.')
        master_df = pd.DataFrame([{'text':'sample legal text','label':'FIR','language':'hindi','source_file':'sample','page':1}])
        master_df.to_csv(DATASET_CSV, index=False)

Building dataset from raw PDFs...
Processing class: FIR


100%|██████████| 193/193 [00:31<00:00,  6.11it/s]


Processing class: Bail_Application


100%|██████████| 1/1 [00:02<00:00,  2.95s/it]


Processing class: Court_Notice


...

DOCX ERROR: /content/drive/MyDrive/NyayaSetu_Dataset/raw-pdfs/Property_Deed/14_Deed_of_conveyance_for_urban_property_land_and_building_pursuant_to_Agreement.doc: file '/content/drive/MyDrive/NyayaSetu_Dataset/raw-pdfs/Property_Deed/14_Deed_of_conveyance_for_urban_property_land_and_building_pursuant_to_Agreement.doc' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'


 57%|█████▋    | 16/28 [06:46<00:23,  1.92s/it]

DOCX ERROR: /content/drive/MyDrive/NyayaSetu_Dataset/raw-pdfs/Property_Deed/15_Deed_of_conveyance_of_urban_land_persuant_to_agreement.doc: file '/content/drive/MyDrive/NyayaSetu_Dataset/raw-pdfs/Property_Deed/15_Deed_of_conveyance_of_urban_land_persuant_to_agreement.doc' is not a Word file, content type is 'application/vnd.openxmlformats-officedocument.themeManager+xml'


100%|██████████| 28/28 [07:04<00:00, 15.18s/it]


Dataset saved: (2415, 5) → /content/drive/MyDrive/NyayaSetu/processed/master_dataset.csv


## 7. MuRIL Legal Document Classifier (Train once, else Load)

In [ ]:
muril_tokenizer = None
muril_model     = None
label_map       = None
id2label        = None

def load_classifier():
    global muril_tokenizer, muril_model, label_map, id2label
    print('Loading MuRIL Classifier from Drive...')
    label_map = load_json(LABEL_MAP_PATH)
    id2label  = {v: k for k, v in label_map.items()}
    muril_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_DIR)
    muril_model = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_DIR).to(DEVICE)
    muril_model.eval()
    print('Classifier loaded successfully.')

def train_classifier():
    global muril_tokenizer, muril_model, label_map, id2label
    print('Downloading base MuRIL from Hugging Face & Training Classifier...')
    labels = sorted(master_df['label'].unique().tolist())
    label_map = {l: i for i, l in enumerate(labels)}
    id2label  = {i: l for l, i in label_map.items()}
    save_json(label_map, LABEL_MAP_PATH)

    master_df['label_id'] = master_df['label'].map(label_map)
    muril_tokenizer = AutoTokenizer.from_pretrained(MURIL_BASE)
    muril_model = AutoModelForSequenceClassification.from_pretrained(
        MURIL_BASE, num_labels=len(labels),
        id2label=id2label, label2id=label_map
    ).to(DEVICE)

    enc = muril_tokenizer(master_df['text'].tolist(), truncation=True,
                          padding='max_length', max_length=MAX_LEN)
    import torch as _t
    class _DS(_t.utils.data.Dataset):
        def __init__(self, enc, labels):
            self.enc = enc; self.labels = labels
        def __len__(self): return len(self.labels)
        def __getitem__(self, i):
            item = {k: _t.tensor(v[i]) for k, v in self.enc.items()}
            item['labels'] = _t.tensor(self.labels[i])
            return item

    ds = _DS(enc, master_df['label_id'].tolist())
    args = TrainingArguments(
        output_dir=CLASSIFIER_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        save_strategy='epoch',
        save_total_limit=1,
        logging_steps=50,
        report_to=[],
    )
    trainer = Trainer(
        model=muril_model, args=args, train_dataset=ds,
        tokenizer=muril_tokenizer, data_collator=DataCollatorWithPadding(muril_tokenizer)
    )
    trainer.train()

    print('Saving fine-tuned classifier to Drive...')
    muril_model.save_pretrained(CLASSIFIER_DIR)
    muril_tokenizer.save_pretrained(CLASSIFIER_DIR)
    muril_model.eval()

try:
    if os.path.exists(CLASSIFIER_DIR) and os.path.exists(os.path.join(CLASSIFIER_DIR, 'config.json')):
        load_classifier()
    else:
        train_classifier()
except Exception as e:
    print(f'Classifier error: {e}')
    raise

## 8. NLLB-200 Translation (Download from HF, cache to Drive)

In [ ]:
nllb_tokenizer = None
nllb_model     = None

NLLB_LANGS = {
    'hindi': 'hin_Deva', 'marathi': 'mar_Deva', 'english': 'eng_Latn',
    'hi': 'hin_Deva', 'mr': 'mar_Deva', 'en': 'eng_Latn',
}

def load_nllb():
    global nllb_tokenizer, nllb_model
    print('Loading NLLB-200...')
    if os.path.exists(NLLB_DIR) and os.path.exists(os.path.join(NLLB_DIR, 'config.json')):
        nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_DIR)
        nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_DIR).to(DEVICE).eval()
        print('NLLB loaded from Drive cache.')
    else:
        print('Downloading NLLB from Hugging Face Hub...')
        nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_BASE)
        nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_BASE).to(DEVICE).eval()
        nllb_tokenizer.save_pretrained(NLLB_DIR)
        nllb_model.save_pretrained(NLLB_DIR)
        print('NLLB downloaded & saved to Drive cache.')

def translate_text(text, src_lang='hindi', tgt_lang='english'):
    src = NLLB_LANGS.get(src_lang.lower(), 'hin_Deva')
    tgt = NLLB_LANGS.get(tgt_lang.lower(), 'eng_Latn')
    nllb_tokenizer.src_lang = src
    inputs = nllb_tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    forced = nllb_tokenizer.lang_code_to_id[tgt]
    with torch.no_grad():
        out = nllb_model.generate(**inputs, forced_bos_token_id=forced, max_length=512)
    return nllb_tokenizer.batch_decode(out, skip_special_tokens=True)[0]

try:
    load_nllb()
except Exception as e:
    print(f'NLLB load error: {e}')

## 9. BART Summarizer (Download from HF, cache to Drive)

In [ ]:
bart_tokenizer = None
bart_model     = None

def load_bart():
    global bart_tokenizer, bart_model
    print('Loading BART...')
    if os.path.exists(BART_DIR) and os.path.exists(os.path.join(BART_DIR, 'config.json')):
        bart_tokenizer = BartTokenizer.from_pretrained(BART_DIR)
        bart_model = BartForConditionalGeneration.from_pretrained(BART_DIR).to(DEVICE).eval()
        print('BART loaded from Drive cache.')
    else:
        print('Downloading BART from Hugging Face Hub...')
        bart_tokenizer = BartTokenizer.from_pretrained(BART_BASE)
        bart_model = BartForConditionalGeneration.from_pretrained(BART_BASE).to(DEVICE).eval()
        bart_tokenizer.save_pretrained(BART_DIR)
        bart_model.save_pretrained(BART_DIR)
        print('BART downloaded & saved to Drive cache.')

def summarize_text(text, max_len=130, min_len=30):
    inputs = bart_tokenizer(text, return_tensors='pt', truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        ids = bart_model.generate(**inputs, max_length=max_len, min_length=min_len,
                                  num_beams=4, length_penalty=2.0, early_stopping=True)
    return bart_tokenizer.decode(ids[0], skip_special_tokens=True)

try:
    load_bart()
except Exception as e:
    print(f'BART load error: {e}')

## 10. SentenceTransformer (Download from HF, cache to Drive)

In [ ]:
st_model = None

def load_st():
    global st_model
    print('Loading SentenceTransformer...')
    if os.path.exists(ST_DIR) and os.path.exists(os.path.join(ST_DIR, 'config.json')):
        st_model = SentenceTransformer(ST_DIR, device=DEVICE)
        print('ST loaded from Drive cache.')
    else:
        print('Downloading SentenceTransformer from Hugging Face Hub...')
        st_model = SentenceTransformer(ST_BASE, device=DEVICE)
        st_model.save(ST_DIR)
        print('ST downloaded & saved to Drive cache.')

def embed_texts(texts):
    if isinstance(texts, str): texts = [texts]
    with torch.no_grad():
        return st_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True,
                               show_progress_bar=False)

try:
    load_st()
except Exception as e:
    print(f'ST load error: {e}')

## 11. Embeddings (build once, else load)

In [ ]:
corpus_embeddings = None
corpus_texts      = None

def build_embeddings():
    global corpus_embeddings, corpus_texts
    print('Building corpus embeddings...')
    corpus_texts = master_df['text'].tolist()
    batch = 64
    all_emb = []
    for i in tqdm(range(0, len(corpus_texts), batch)):
        all_emb.append(embed_texts(corpus_texts[i:i+batch]))
    corpus_embeddings = np.vstack(all_emb).astype('float32')
    np.save(EMB_PATH, corpus_embeddings)
    save_json({'count': len(corpus_texts)}, META_PATH)
    print('Embeddings saved to', EMB_PATH)

def load_embeddings():
    global corpus_embeddings, corpus_texts
    print('Loading embeddings from Drive...')
    corpus_embeddings = np.load(EMB_PATH).astype('float32')
    corpus_texts = master_df['text'].tolist()
    print('Embeddings shape:', corpus_embeddings.shape)

try:
    if os.path.exists(EMB_PATH):
        load_embeddings()
    else:
        build_embeddings()
except Exception as e:
    print(f'Embedding error: {e}')
    raise

## 12. FAISS Index (build once, else load)

In [ ]:
faiss_index = None

def build_faiss():
    global faiss_index
    print('Building FAISS index...')
    dim = corpus_embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(corpus_embeddings)
    faiss.write_index(faiss_index, FAISS_INDEX_PATH)
    print('FAISS index saved to', FAISS_INDEX_PATH)

def load_faiss():
    global faiss_index
    print('Loading FAISS index from Drive...')
    faiss_index = faiss.read_index(FAISS_INDEX_PATH)
    print('FAISS total vectors:', faiss_index.ntotal)

try:
    if os.path.exists(FAISS_INDEX_PATH):
        load_faiss()
    else:
        build_faiss()
except Exception as e:
    print(f'FAISS error: {e}')
    raise

## 13. BM25 Index (build once, else load)

In [ ]:
bm25 = None

def _tokenize(s):
    return re.findall(r'\w+', s.lower())

def build_bm25():
    global bm25
    print('Building BM25 index...')
    tokenized = [_tokenize(t) for t in corpus_texts]
    bm25 = BM25Okapi(tokenized)
    with open(BM25_PATH, 'wb') as f:
        pickle.dump(bm25, f)
    print('BM25 saved to', BM25_PATH)

def load_bm25():
    global bm25
    print('Loading BM25 index from Drive...')
    with open(BM25_PATH, 'rb') as f:
        bm25 = pickle.load(f)

try:
    if os.path.exists(BM25_PATH):
        load_bm25()
    else:
        build_bm25()
except Exception as e:
    print(f'BM25 error: {e}')
    raise

## 14. Hybrid Retrieval & Metadata

In [ ]:
def hybrid_retrieve(query, top_k=5, alpha=0.5):
    if not query or not corpus_texts:
        return []
    q_emb = embed_texts(query)[0].astype('float32').reshape(1, -1)
    faiss_scores, faiss_ids = faiss_index.search(q_emb, top_k * 4)
    bm25_scores = bm25.get_scores(_tokenize(query))
    faiss_norm = (faiss_scores[0] - faiss_scores[0].min()) / (faiss_scores[0].max() - faiss_scores[0].min() + 1e-8)
    candidates = {}
    for rank, idx in enumerate(faiss_ids[0]):
        candidates[idx] = alpha * float(faiss_norm[rank])
    top_bm = np.argsort(bm25_scores)[::-1][:top_k * 4]
    bm_arr = bm25_scores[top_bm]
    bm_norm = (bm_arr - bm_arr.min()) / (bm_arr.max() - bm_arr.min() + 1e-8)
    for rank, j in enumerate(top_bm):
        candidates[int(j)] = candidates.get(int(j), 0.0) + (1 - alpha) * float(bm_norm[rank])
    best = sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:top_k]
    results = []
    for idx, score in best:
        row = master_df.iloc[idx]
        results.append({
            'text': row['text'][:500],
            'label': row['label'],
            'language': row.get('language', 'unknown'),
            'source_file': row.get('source_file', ''),
            'page': int(row.get('page', 0)),
            'score': float(score)
        })
    return results

print('Metadata in memory: corpus size =', len(corpus_texts) if corpus_texts else 0)

## 15. Initialize FastAPI (all endpoints preserved)

In [ ]:
app = FastAPI(title='NyayaSetu Legal AI API')

class TranslateReq(BaseModel):
    text: str
    src_lang: str = 'hindi'
    tgt_lang: str = 'english'

class SummarizeReq(BaseModel):
    text: str
    max_length: int = 130
    min_length: int = 30

class EmbedReq(BaseModel):
    texts: List[str]

class ClassifyReq(BaseModel):
    text: str

class RetrieveReq(BaseModel):
    query: str
    top_k: int = 5
    alpha: float = 0.5

@app.get('/health')
def health():
    return {'status':'ok','device':DEVICE,
            'faiss_vectors': int(faiss_index.ntotal) if faiss_index else 0,
            'corpus_size': len(corpus_texts) if corpus_texts else 0}

@app.post('/translate')
def translate(req: TranslateReq):
    try:
        return {'translated': translate_text(req.text, req.src_lang, req.tgt_lang)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post('/summarize')
def summarize(req: SummarizeReq):
    try:
        return {'summary': summarize_text(req.text, req.max_length, req.min_length)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post('/embed')
def embed(req: EmbedReq):
    try:
        vecs = embed_texts(req.texts)
        return {'embeddings': vecs.tolist()}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post('/classify')
def classify(req: ClassifyReq):
    try:
        inputs = muril_tokenizer(req.text, return_tensors='pt', truncation=True,
                                padding=True, max_length=MAX_LEN).to(DEVICE)
        with torch.no_grad():
            logits = muril_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        idx = int(torch.argmax(probs))
        return {'label': id2label[idx], 'confidence': float(probs[idx])}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post('/retrieve')
def retrieve(req: RetrieveReq):
    try:
        return {'results': hybrid_retrieve(req.query, req.top_k, req.alpha)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print('FastAPI app initialized with 6 endpoints: /health /translate /summarize /embed /classify /retrieve')

## 16. ngrok (authenticate, reuse tunnel if exists)

In [8]:
public_url = None

def start_ngrok(port=8000):
    global public_url
    if NGROK_TOKEN:
        ngrok.set_auth_token(NGROK_TOKEN)
    # Reuse existing tunnel if present
    for t in ngrok.get_tunnels():
        if str(port) in t.public_url or 'http' in t.public_url:
            public_url = t.public_url
            print(f'Reusing existing ngrok tunnel: {public_url}')
            return public_url
    try:
        public_url = ngrok.connect(port)
        print('ngrok public URL:', public_url)
    except Exception as e:
        print(f'ngrok error: {e}')
        print('Set NGROK_TOKEN in the config cell.')
    return public_url

start_ngrok(8000)

ngrok public URL: NgrokTunnel: "https://immorally-coveted-arguably.ngrok-free.dev" -> "http://localhost:8000"


<NgrokTunnel: "https://immorally-coveted-arguably.ngrok-free.dev" -> "http://localhost:8000">

## 17. Start Uvicorn Server (background)

In [9]:
import threading, socket

def start_server(port=8000):
    config = uvicorn.Config(app, host='0.0.0.0', port=port, log_level='info')
    server = uvicorn.Server(config)
    server.run()

def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

if not _port_in_use(8000):
    t = threading.Thread(target=start_server, daemon=True)
    t.start()
    time.sleep(3)
    print('Uvicorn started in background.')
else:
    print('Uvicorn already running on port 8000.')

if public_url:
    print('=' * 60)
    print('NyayaSetu API is live at:', public_url)
    print('Endpoints: /health /translate /summarize /embed /classify /retrieve')
    print('=' * 60)
else:
    print('Server running locally on http://localhost:8000 (no ngrok URL)')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Exception in thread Thread-5 (start_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_2030/2309060525.py", line 4, in start_server
NameError: name 'app' is not defined


Uvicorn started in background.
NyayaSetu API is live at: NgrokTunnel: "https://immorally-coveted-arguably.ngrok-free.dev" -> "http://localhost:8000"
Endpoints: /health /translate /summarize /embed /classify /retrieve


## 18. Testing

In [ ]:
import requests

BASE = str(public_url).rstrip('/') if public_url else 'http://localhost:8000'

print('Health:', requests.get(f'{BASE}/health').json())

sample = 'जमानत अर्जदार व समोरच्या पक्षाच्या वकिलांनी याबाबत आपापले म्हणणे मांडले.'
print('Translate:', requests.post(f'{BASE}/translate', json={'text': sample, 'src_lang':'marathi','tgt_lang':'english'}).json())

eng = ('This is a sample legal document about property dispute between two parties '
       'regarding the ownership of land located in the district. The plaintiff claims '
       'rightful ownership based on ancestral records.')
print('Summarize:', requests.post(f'{BASE}/summarize', json={'text': eng}).json())

print('Embed:', requests.post(f'{BASE}/embed', json={'texts':[eng]}).json()['embeddings'][0][:5])

print('Classify:', requests.post(f'{BASE}/classify', json={'text': eng}).json())

print('Retrieve:', requests.post(f'{BASE}/retrieve', json={'query': eng, 'top_k': 3}).json())